# 🔬 Skin Cancer Detection — EDA & Model Training
**Minor Project | Deep Learning | Transfer Learning (MobileNetV2)**

### Dataset: HAM10000
- 10,015 dermoscopic images
- 7 classes of skin lesions
- Source: https://dataverse.harvard.edu/dataset.xhtml?persistentId=doi:10.7910/DVN/DBW86T


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import os, random

sns.set_theme(style='darkgrid')
plt.rcParams['figure.dpi'] = 100
print('Libraries loaded ✅')

## 1. Load & Explore Dataset

In [ ]:
df = pd.read_csv('../dataset/HAM10000_metadata.csv')
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

In [ ]:
CLASS_NAMES = {
    'nv':    'Melanocytic Nevi',
    'mel':   'Melanoma',
    'bkl':   'Benign Keratosis',
    'bcc':   'Basal Cell Carcinoma',
    'akiec': 'Actinic Keratoses',
    'vasc':  'Vascular Lesions',
    'df':    'Dermatofibroma'
}

# Class distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

counts = df['dx'].value_counts()
labels = [CLASS_NAMES[c] for c in counts.index]
colors = ['#e74c3c','#e67e22','#f1c40f','#2ecc71','#3498db','#9b59b6','#1abc9c']

axes[0].bar(labels, counts.values, color=colors)
axes[0].set_title('Class Distribution (Count)', fontsize=13, fontweight='bold')
axes[0].tick_params(axis='x', rotation=30)

axes[1].pie(counts.values, labels=labels, colors=colors, autopct='%1.1f%%', startangle=140)
axes[1].set_title('Class Distribution (%)', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()
print('Note: Dataset is highly IMBALANCED — nv (nevi) dominates at ~67%')

In [ ]:
# Age, sex, localization analysis
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

df['age'].hist(ax=axes[0], bins=20, color='#3498db', edgecolor='white')
axes[0].set_title('Age Distribution')
axes[0].set_xlabel('Age')

df['sex'].value_counts().plot(kind='bar', ax=axes[1], color=['#e74c3c','#3498db','#95a5a6'])
axes[1].set_title('Gender Distribution')
axes[1].tick_params(axis='x', rotation=0)

df['localization'].value_counts().head(8).plot(kind='barh', ax=axes[2], color='#9b59b6')
axes[2].set_title('Top 8 Lesion Locations')

plt.tight_layout()
plt.show()

## 2. Sample Images per Class

In [ ]:
IMG_DIR = '../dataset/HAM10000_images'

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, (cls_key, cls_name) in enumerate(CLASS_NAMES.items()):
    subset = df[df['dx'] == cls_key]
    sample = subset.sample(1).iloc[0]
    img_path = os.path.join(IMG_DIR, sample['image_id'] + '.jpg')
    if os.path.exists(img_path):
        img = Image.open(img_path)
        axes[i].imshow(img)
        axes[i].set_title(f'{cls_name}\n({cls_key})', fontsize=9, fontweight='bold')
        axes[i].axis('off')

axes[-1].axis('off')
plt.suptitle('Sample Images from Each Class', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Build & Train the Model

In [ ]:
import sys
sys.path.append('..')

# Simply run the training script
# OR execute the steps below for more control
import subprocess
result = subprocess.run(['python', '../train_model.py'], capture_output=True, text=True)
print(result.stdout[-3000:])

## 4. Evaluate Results

In [ ]:
# Show saved plots
from IPython.display import Image as IPyImage, display

print('Training History:')
display(IPyImage('../static/training_history.png'))
print('\nConfusion Matrix:')
display(IPyImage('../static/confusion_matrix.png'))

## 5. Single Image Prediction

In [ ]:
import tensorflow as tf
import numpy as np
from PIL import Image

model = tf.keras.models.load_model('../models/skin_cancer_model.h5')

def predict_image(img_path):
    img = Image.open(img_path).convert('RGB').resize((224, 224))
    arr = np.array(img) / 255.0
    arr = np.expand_dims(arr, 0)
    preds = model.predict(arr)[0]
    classes = list(CLASS_NAMES.keys())
    pred_idx = np.argmax(preds)
    
    print(f'Predicted: {CLASS_NAMES[classes[pred_idx]]} ({classes[pred_idx]})')
    print(f'Confidence: {preds[pred_idx]*100:.1f}%\n')
    
    plt.figure(figsize=(8,3))
    plt.subplot(1,2,1)
    plt.imshow(Image.open(img_path))
    plt.title('Input Image'); plt.axis('off')
    plt.subplot(1,2,2)
    colors = ['red' if i == pred_idx else '#3498db' for i in range(7)]
    plt.barh([CLASS_NAMES[c] for c in classes], preds*100, color=colors)
    plt.xlabel('Probability (%)')
    plt.title('Class Probabilities')
    plt.tight_layout(); plt.show()

# predict_image('path/to/your/image.jpg')